# 💾 Download Images under 20minutes instead of 4hours

As the original script is not optimized, I have updated it to speed up the download from ~4hours to 20 minutes. 
Below you can find the approach ffor directl use on Kaggle and code snippet ready for copy paste into your project.

**Authors:**
- Eric Orenstein (eorenstein@mbari.org)
- Lukas Picek (lukaspicek@gmail.com)


# 🏃 Run this to retrive the images on Kaggle


⚠️ There is not enough space on Kaggle to store the images in original resolution. In that case, you have to resize it before saving. Just set `resize = True` ⚠️

In [1]:
import os
import json
import logging
import argparse
import requests
import progressbar
from PIL import Image
from tqdm import tqdm
from shutil import copyfileobj
from multiprocessing import Pool, cpu_count


def download_img(args, resize=True):
    """
    Download a single image.
    
    :param args: Tuple of (name, url, outdir)
    """
    name, url, outdir = args
    file_name = os.path.join(outdir, name)

    # Only download if the image does not exist in the outdir
    if not os.path.exists(file_name):
        resp = requests.get(url, stream=True)
        resp.raw.decode_content = True
        with open(file_name, 'wb') as f:
            copyfileobj(resp.raw, f)
        
        # Resize the downloaded image while keeping the aspect ratio
        if resize:
            try:
                img = Image.open(file_name)
                img.thumbnail((1280,720), Image.LANCZOS)
                img.save(file_name)
            except:
                print(f"Problem with resizing image: {file_name}")
        
        return 1  # Indicate that image was downloaded
    else:
        return 0  # Indicate that image already exists


def download_imgs(imgs, outdir=None):
    """
    Download images to an output dir
    
    :param imgs: list of tuples (name, url)
    :param outdir: desired directory [default to working directory]
    """

    # Set the out directory to default if not specified
    if not outdir:
        outdir = os.path.join(os.getcwd(), 'images')

    # Make the directory if it does not exist
    if not os.path.exists(outdir):
        os.mkdir(outdir)
        logging.info(f"Created directory {outdir}")

    num_processes = cpu_count() * 2  # Use twice the number of CPU cores for multiprocessing
    pool = Pool(processes=num_processes)

    # Prepare arguments for multiprocessing
    args_list = [(name, url, outdir) for name, url in imgs]

    # Use tqdm for progress bar
    with tqdm(total=len(imgs)) as pbar:
        for _ in pool.imap_unordered(download_img, args_list):
            pbar.update(1)

    pool.close()
    pool.join()

In [ ]:
dataset = "train.json"
outpath = "kaggle\\training-data"

logging.info(f'opening {dataset}')
with open(dataset, 'r') as ff:
    dataset = json.load(ff)

ims = dataset['images']

logging.info(f'retrieving {len(ims)} images')

ims = [(im['file_name'], im['coco_url']) for im in ims]

# Download images
download_imgs(ims, outdir=outpath)

  0%|          | 0/8058 [00:00<?, ?it/s]

In [ ]:
dataset = "eval.json"
outpath = "kaggle\\test-data"

logging.info(f'opening {dataset}')
with open(dataset, 'r') as ff:
    dataset = json.load(ff)

ims = dataset['images']

logging.info(f'retrieving {len(ims)} images')

ims = [(im['file_name'], im['coco_url']) for im in ims]

# Download images
download_imgs(ims, outdir=outpath)